## High-Speed Rail Analysis

### CRISP-DM Step 1: Business Understanding

Goal: identify all city pairs where a high-speed rail line exists, then determine how many of these routes are faster than taking air travel.

## Imports

In [5]:
import pandas as pd
import geopandas as gpd
import osmium
from shapely.geometry import LineString
import re
from pathlib import Path

### CRISP-DM Step 2: Data Understanding

Goal: collect data from OpenStreetMap, describe its structure, and assess quality.

The .osm.pbf files for all EU-27 countries must be downloaded from https://download.geofabrik.de/europe.html. The files must then be dropped into ../Raw_Data/OSM_data/

#### 2.1 Data Collection (Data Loading)

In [28]:
class RailwayExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.features = []

    def way(self, w):
        if w.tags.get("railway") != "rail":
            return

        coords = [(n.lon, n.lat) for n in w.nodes if n.location.valid()]
        if len(coords) < 2:
            return

        self.features.append({
            "geometry": LineString(coords),
            "maxspeed": w.tags.get("maxspeed"),
            "service": w.tags.get("service"),
            "usage": w.tags.get("usage"),
            "name": w.tags.get("name"),
        })

def extract_railways(pbf_path):
    handler = RailwayExtractor()
    handler.apply_file(pbf_path, locations=True)

    gdf = gpd.GeoDataFrame(
        handler.features,
        geometry="geometry",
        crs="EPSG:4326"
    )

    return gdf

def parse_maxspeed(val):
    if val is None:
        return None

    s = str(val).lower()

    m = re.search(r"\d+", str(val))
    if not m:
        return None

    speed = float(m.group())
    # Convert mph → km/h
    if "mph" in s:
        speed *= 1.609344
    return speed

def mark_highspeed(gdf):
    gdf = gdf.copy()

    gdf["maxspeed_num"] = gdf["maxspeed"].apply(parse_maxspeed)

    gdf["is_highspeed"] = (
        (gdf["maxspeed_num"] >= 200)
    )

    return gdf

osm_path =  Path('..') / 'Raw_Data' / 'OSM_Data'
output_dir = Path('..') / 'Raw_Data' / 'Parquet_Data'
output_dir.mkdir(exist_ok=True)

for pbf in osm_path.glob("*.osm.pbf"):
    print(f"Processing {pbf.name}")

    gdf = extract_railways(pbf)
    gdf = mark_highspeed(gdf)

    out = output_dir / f"{pbf.stem}_rail.parquet"
    gdf.to_parquet(out)

Processing spain-260101.osm.pbf
Processing denmark-260101.osm.pbf
Processing greece-260101.osm.pbf
Processing estonia-260101.osm.pbf
Processing lithuania-260101.osm.pbf
Processing france-260101.osm.pbf
Processing austria-260101.osm.pbf
Processing monaco-260101.osm.pbf
Processing portugal-260101.osm.pbf
Processing romania-260101.osm.pbf
Processing sweden-260101.osm.pbf
Processing latvia-260101.osm.pbf
Processing liechtenstein-260101.osm.pbf
Processing belgium-260101.osm.pbf
Processing netherlands-260101.osm.pbf
Processing germany-latest.osm.pbf
Processing italy-260101.osm.pbf
Processing poland-260101.osm.pbf
Processing bulgaria-260101.osm.pbf
Processing hungary-260101.osm.pbf
Processing finland-260101.osm.pbf
Processing slovenia-260101.osm.pbf
Processing czech-republic-260101.osm.pbf
Processing ireland-and-northern-ireland-260101.osm.pbf
Processing luxembourg-260101.osm.pbf
Processing croatia-260101.osm.pbf
Processing slovakia-260101.osm.pbf


##### Merge all parquet files into one

In [29]:
files = output_dir.glob("*.parquet")

gdfs = [gpd.read_parquet(f) for f in files]

merged = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

merged_path = Path('..') / 'Processed_Data' / 'EU_railways.parquet'
merged.to_parquet(merged_path)

In [30]:
railways = gpd.read_parquet(merged_path)
print(railways.head)
print(railways[railways['is_highspeed']].head) # filter for is_highspeed

<bound method NDFrame.head of                                                  geometry maxspeed service  \
0       LINESTRING (-6.24157 53.34133, -6.2423 53.3416...       30    None   
1       LINESTRING (-8.65163 52.34925, -8.64136 52.359...   90 mph    None   
2       LINESTRING (-9.49296 52.05867, -9.49438 52.058...   15 mph    None   
3       LINESTRING (-7.8216 52.67822, -7.821 52.68128,...      160    None   
4       LINESTRING (-7.8235 52.78557, -7.82302 52.7867...      160    None   
...                                                   ...      ...     ...   
890261    LINESTRING (23.35432 54.328, 23.35571 54.32808)      100    None   
890262  LINESTRING (23.35571 54.32808, 23.35686 54.32814)      100    None   
890263     LINESTRING (23.38484 55.9119, 23.386 55.91151)     None    yard   
890264  LINESTRING (23.38685 55.91078, 23.38719 55.910...     None    yard   
890265  LINESTRING (21.23051 55.8913, 21.23041 55.8906...      100  siding   

       usage              name  m

In [ ]:
routes_path = Path('..') / 'Processed_Data' / 'train_routes_scraped_no_duplicates.csv'
stations_path = Path('..') / 'Processed_Data' / 'city_train_stations.csv'

routes = pd.read_csv(routes_path)
stations = pd.read_csv(stations_path)